# search-compare — Exemplo de uso

Este notebook demonstra como usar a biblioteca `search_compare` para:

1. Construir duas strings de busca PICOC
2. Executar as buscas na Scopus
3. Comparar os resultados e identificar documentos em comum, exclusivos de cada string

## Pré-requisitos

Defina a variável de ambiente `SCOPUS_API_KEY` antes de executar.

Ou crie um arquivo `.env` na raiz do projeto com sua chave da API Scopus:

```
SCOPUS_API_KEY=sua_chave_aqui
```


## 1. Imports

In [1]:
import sys
sys.path.insert(0, "../src")  # necessário apenas quando executado fora do ambiente instalado

from search_compare import (
    QueryBuilder,
    ScopusClient,
    ScopusQueryBuilder,
    compare,
)

## 2. Configurar os filtros Scopus

O `ScopusQueryBuilder` é compartilhado entre os dois builders e mantém os filtros configurados.

Opções disponíveis em `configure`:
- `field`: `"TITLE-ABS-KEY"` (padrão), `"TITLE"`, `"ABS"`, `"KEY"`
- `year_min` / `year_max`: intervalo de publicação (exclusivo)
- `language`: ex. `"english"`
- `doc_type`: `"ar"` (artigo), `"re"` (review), `"cp"` (conference paper), etc.

In [2]:
scopus_qb = ScopusQueryBuilder()
scopus_qb.configure(
    field="TITLE-ABS-KEY",
    year_min=2020,
    year_max=2026,
    language="english",
)

## 3. Construir as strings de busca PICOC

Cada componente PICOC aceita múltiplos termos, que são combinados com `OR`.
Os componentes definidos são combinados com `AND`.

Termos com espaços devem estar entre aspas: `'"embedded system"'`

In [ ]:
b1 = QueryBuilder(backend=scopus_qb)
b1.population('"embedded software"', '"embedded system"', 'microcontroller')
b1.intervention('architecture', '"design pattern"')
b1.outcome('maintainability', 'maintainable', 'testability', 'testable')

query1 = b1.build()
print("Query 1 (core):")
print(b1.build_core())
print("\nQuery 1 (completa):")
print(query1)

Query 1 (core):
("embedded software" OR "embedded system" OR microcontroller) AND (architecture OR "design pattern") AND (maintainability OR maintainable OR testability OR testable)

Query 1 (completa):
TITLE-ABS-KEY(("embedded software" OR "embedded system" OR microcontroller) AND (architecture OR "design pattern") AND (maintainability OR maintainable OR testability OR testable)) AND PUBYEAR > 2020 AND PUBYEAR < 2026 AND LANGUAGE(english)


In [ ]:
b2 = QueryBuilder(backend=scopus_qb)
b2.population('MCU', 'microcontroller', '"embedded software"', '"embedded system"')
b2.intervention('architecture', '"design pattern"')
b2.outcome('SOLID', 'extensibility', 'extensible', 'maintainability', 'maintainable', 'testability', 'testable')
b2.context('IoT', '"internet of things"', '"Cyber-Physical Systems"')

query2 = b2.build()
print("Query 2 (core):")
print(b2.build_core())
print("\nQuery 2 (completa):")
print(query2)

Query 2 (core):
(MCU OR microcontroller OR "embedded software" OR "embedded system") AND (architecture OR "design pattern") AND (SOLID OR extensibility OR extensible OR maintainability OR maintainable OR testability OR testable) AND (IoT OR "internet of things" OR "Cyber-Physical Systems")

Query 2 (completa):
TITLE-ABS-KEY((MCU OR microcontroller OR "embedded software" OR "embedded system") AND (architecture OR "design pattern") AND (SOLID OR extensibility OR extensible OR maintainability OR maintainable OR testability OR testable) AND (IoT OR "internet of things" OR "Cyber-Physical Systems")) AND PUBYEAR > 2020 AND PUBYEAR < 2026 AND LANGUAGE(english)


## 4. Executar as buscas

In [5]:
client = ScopusClient()  # lê SCOPUS_API_KEY do .env ou variável de ambiente

print("Buscando resultados para Query 1...")
result1 = client.search(query1, max_results=500)
print(f"  Recuperados: {len(result1)} / Total disponível: {result1.total_count}")

print("Buscando resultados para Query 2...")
result2 = client.search(query2, max_results=500)
print(f"  Recuperados: {len(result2)} / Total disponível: {result2.total_count}")

Buscando resultados para Query 1...


  Recuperados: 74 / Total disponível: 74
Buscando resultados para Query 2...


  Recuperados: 102 / Total disponível: 102


## 5. Comparar os resultados

In [6]:
diff = compare(result1, result2)
print(diff.summary())

Total in query 1 : 74
Total in query 2 : 102
Common           : 23
Only in query 1  : 51
Only in query 2  : 79


## 6. Visualizar os resultados

In [7]:
diff.show()

### Documentos em comum (23)

,year,title,authors,source
0,2024,Intelligent Turning Cyber-Physical Systems Modeling using SysML,Dey P.R.,IECON Proceedings Industrial Electronics Conference
1,2025,Design and Performance Comparison of Current References for a 32-bit Microcontroller Using 22nm Technology,Barcelos E.A.,2025 IEEE 16th Latin American Symposium on Circuits and Systems Lascas 2025 Proceedings
2,2025,Interoperability Assessment Oriented Towards the Development of Digital Twin Architecture in Industrial Maintenance,Junior R.P.L.,Lecture Notes in Production Engineering
3,2025,An Intelligent Gamified Reviewer System for Enhancing Student Engagement and Performance in Cyber-Physical Learning Environments,Saballa-Marin M.J.M.,International Conference on Communication Computing Networking and Control in Cyber Physical Systems Ccncps 2025
4,2025,"A CPS-Based Architecture for Mobile Robotics: Design, Integration, and Localisation Experiments",Líšková D.,Sensors
5,2025,Lightweight Embedded IoT Gateway for Smart Homes Based on an ESP32 Microcontroller,Serepas F.,Computers
6,2025,Component-based Control Software design using IEC 61499 Adapter Interfaces,Sharma S.,IEEE International Conference on Emerging Technologies and Factory Automation ETFA
7,2025,Multivariate Time Series Anomaly Detection in Cyber-Physical Systems Using Sparse Attention,Li Y.,IECON Proceedings Industrial Electronics Conference
8,2025,Development Of A Cloud Portal For Process-Oriented Programming Toolkits,Epov I.,Proceedings of the 17th International Scientific and Technical Conference Actual Problems of Electronic Instrument Engineering Apeie 2025
9,2025,Rainmaker-rs: A Cross-Platform Rust Implementation of RainMaker for ESP and Linux Devices,Bubane S.,2025 9th International Conference on Computing Communication Control and Automation Icccbea 2025


### Apenas na query 1 (51)

,year,title,authors,source
0,2024,The embedded project cookbook: A step-by-step guide for microcontroller projects,Taylor J.T.,Embedded Project Cookbook A Step by Step Guide for Microcontroller Projects
1,2024,Robustness Methodology for Next Generation Automotive Microcontroller Flip Chip Copper Pillar Technologies,Chong T.A.,Proceedings of the 26th Electronics Packaging Technology Conference EPTC 2024
2,2024,Enhancing Mobility with Next-Generation Partial Hand Prosthesis Using Arduino Uno and Flex Sensors,Matury S.S.,Proceedings of the 2024 2nd International Conference on Cyber Physical Systems Power Electronics and Electric Vehicles Icpeev 2024
3,2024,Implementation of a Watchdog timer's Microarchitecture and design for a RISC-V-based SoC using 45nm Technology,Chaithanya D.J.,2024 Asian Conference on Intelligent Technologies Acoit 2024
4,2024,Architecture description languages,Chattopadhyay A.,Handbook of Computer Architecture
5,2025,TMI: Two-dimensional maintainability index for automotive software maintainability measurement,Xie J.,Journal of Systems Architecture
6,2025,AHA: Design and Evaluation of Compute-Intensive Hardware Accelerators for AMD-Xilinx Zynq SoCs Using HLS IP Flow,Berrazueta-Mena D.,Computers
7,2025,Control Logic Synthesis: Drawing the Rest of the OWL,Sisco Z.D.,International Conference on Architectural Support for Programming Languages and Operating Systems ASPLOS
8,2025,Architecting Test-Friendly Multi-Mode Scan Flip-Flops for Modern Low-Power Digital Systems,Kumar T.R.D.,International Conference on Trends in Engineering Systems and Technologies Ictest 2025 Proceedings
9,2025,Verification of AMBA APB Bus Protocol Using UVM,Saravanan S.,Proceedings 3rd International Conference on Artificial Intelligence and Machine Learning Applications Healthcare and Internet of Things Aimla 2025


### Apenas na query 2 (79)

,year,title,authors,source
0,2024,An Intelligent Solid Waste Classification and Monitoring Alert System using Deep Learning,Selvi S.,2024 International Conference on Integration of Emerging Technologies for the Digital World Icietdw 2024
1,2025,Real-Time Wireless Emergency Communication for Hazardous Environments,Ganesh C.S.S.,Proceedings of 7th International Conference on Inventive Material Science and Applications Icima 2025
2,2025,Optimizing Robotic Disassembly-Assembly Line Balancing with Directional Switching Time via an Improved Q(λ) Algorithm in IoT-Enabled Smart Manufacturing,Zhang Q.,Electronics Switzerland
3,2025,Model of an Open-Source MicroPython Library for GSM NB-IoT,Lupandin A.,Sensors
4,2025,EGaIn-Based Liquid Antennas: Beam Steering and Frequency Reconfiguration Using Microfluidic Control,Spyros L.,Proceedings of IEEE Computer Society Annual Symposium on VLSI Isvlsi
5,2025,Smart Waste Management System an IoT Driven Framework,Chandra L.,2025 6th International Conference on Data Intelligence and Cognitive Informatics Icdici 2025
6,2025,Design and implementation of a secure IoT dashboard for cold room management using penetration testing methods,Lajnef H.,Engineering Research Express
7,2025,RISC-V: A Comprehensive Overview of an Emerging ISA for the AI-IoT Era,Hassan Q.F.,Advances in the Internet of Things Challenges Solutions and Emerging Technologies
8,2025,Deployment of a Machine Learning-SDN Pipeline for IoT Threat Detection in Smart City Environments,Antonio A.W.,IEEE Workshop on Local and Metropolitan Area Networks
9,2025,Benchmarking WebAssembly for Embedded Systems,Moron K.,ACM Transactions on Architecture and Code Optimization
